# FDS (Fraud Detection System)

# 1. Data Cleansing

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

# Performance
import time
from tqdm import tqdm

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1.2 Load Data

In [ ]:
# Baca file Excel
df_raw = pd.

print(f"Total data: {len(df_raw):,} baris")
print(f"Total kolom: {len(df_raw.columns)} kolom")
print("\n=== Kolom yang tersedia ===")
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i}. {col}")
    
print("\n=== Preview 5 data pertama ===")
df_raw.head()

## 1.3 Data Analysis

In [ ]:
# Cek info data
print("=== INFO DATA ===")
print(df.info())

print("\n=== STATISTIK DESKRIPTIF ===")
print(df.describe())

print("\n=== TIPE DATA ===")
print(df.dtypes)

## 1.4 Check Missing Values

In [ ]:
# Cek missing values per kolom
print("=== MISSING VALUES ===")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Kolom': missing_values.index,
    'Jumlah Missing': missing_values.values,
    'Persentase (%)': missing_percentage.values
})

missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Jumlah Missing', ascending=False)

if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("✓ Tidak ada missing values!")

print(f"\nTotal missing values: {df.isnull().sum().sum():,}")
print(f"Persentase total missing: {(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%")

## 1.5 Check Duplicate Data

In [ ]:
# Cek duplikat data
print("=== DUPLIKAT DATA ===")
duplicates = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicates:,}")
print(f"Persentase duplikat: {(duplicates / len(df) * 100):.2f}%")

if duplicates > 0:
    # Cek duplikat berdasarkan transaction_id
    duplicates_by_id = df.duplicated(subset=['transaction_id']).sum()
    print(f"\nDuplikat berdasarkan transaction_id: {duplicates_by_id:,}")
    
    # Lihat contoh data duplikat
    print("\n=== Sample Data Duplikat ===")
    print(df[df.duplicated(keep=False)].head(10))

## 1.6 Data Cleaning

In [ ]:
# Simpan jumlah data sebelum cleaning
data_before = len(df)
print(f"Data sebelum cleaning: {data_before:,} baris")

# Hapus duplikat berdasarkan semua kolom (keep='first' untuk mempertahankan data pertama)
df_clean = df.drop_duplicates(keep='first')

data_after_dup = len(df_clean)
removed_dup = data_before - data_after_dup

print(f"\nData setelah hapus duplikat: {data_after_dup:,} baris")
print(f"Duplikat yang dihapus: {removed_dup:,} baris ({(removed_dup/data_before*100):.2f}%)")

# Cek duplikat transaction_id (seharusnya unique)
dup_trans_id = df_clean['id'].duplicated().sum()
if dup_trans_id > 0:
    print(f"\n⚠️ Warning: Masih ada {dup_trans_id:,} transaction_id yang duplikat!")
else:
    print("\n✓ Semua transaction_id sudah unique!")

## 1.7 Handle Missing Values

In [ ]:
# Cek missing values setelah hapus duplikat
print("=== MISSING VALUES SETELAH HAPUS DUPLIKAT ===")
missing_values = df_clean.isnull().sum()
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100

missing_summary = pd.DataFrame({
    'Kolom': missing_values.index,
    'Jumlah Missing': missing_values.values,
    'Persentase (%)': missing_percentage.values
})

missing_summary = missing_summary[missing_summary['Jumlah Missing'] > 0].sort_values('Jumlah Missing', ascending=False)

if len(missing_summary) > 0:
    print(missing_summary.to_string(index=False))
    print(f"\nTotal missing values: {df_clean.isnull().sum().sum():,}")
else:
    print("✓ Tidak ada missing values!")
    
# Cek distribusi missing values per baris
rows_with_missing = df_clean.isnull().any(axis=1).sum()
print(f"\nJumlah baris dengan missing values: {rows_with_missing:,} ({(rows_with_missing/len(df_clean)*100):.2f}%)")

## 1.8 More Verification

In [ ]:
# Verifikasi final setelah cleaning
print("=== VERIFIKASI DATA CLEAN ===\n")

# 1. Cek missing values
total_missing = df_clean.isnull().sum().sum()
print(f"1. Total missing values: {total_missing:,}")
if total_missing == 0:
    print("   ✓ Tidak ada missing values!")
else:
    print(f"   ⚠️ Masih ada missing values di:")
    print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# 2. Cek duplikat
total_duplicates = df_clean.duplicated().sum()
print(f"\n2. Total duplikat: {total_duplicates:,}")
if total_duplicates == 0:
    print("   ✓ Tidak ada duplikat!")
    
# 3. Cek transaction_id unique
unique_trans_id = df_clean['transaction_id'].nunique()
total_rows = len(df_clean)
print(f"\n3. Transaction ID:")
print(f"   - Total baris: {total_rows:,}")
print(f"   - Unique transaction_id: {unique_trans_id:,}")
if unique_trans_id == total_rows:
    print("   ✓ Semua transaction_id unique!")
else:
    print(f"   ⚠️ Ada {total_rows - unique_trans_id:,} transaction_id yang tidak unique!")

# 4. Ringkasan data
print(f"\n4. Ringkasan Data:")
print(f"   - Total transaksi: {len(df_clean):,}")

print("\n" + "="*50)
print("✅ DATA CLEANING SELESAI!")

## 1.9 Last Preview of Data Cleansing

In [ ]:
df_clean.head(10)

# 2. Feature Engineering

## 2.1 Add More Time-Based Columns

In [ ]:
# Create copy for feature engineering
df_fe = df.copy()

# a. Extract hour, day, month, year
df_fe['hour'] = df_fe['transaction_timestamp'].dt.hour
df_fe['day'] = df_fe['transaction_timestamp'].dt.day
df_fe['month'] = df_fe['transaction_timestamp'].dt.month
df_fe['year'] = df_fe['transaction_timestamp'].dt.year
df_fe['day_of_week'] = df_fe['transaction_timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df_fe['day_of_year'] = df_fe['transaction_timestamp'].dt.dayofyear

# b. Hari libur nasional Indonesia (Juli - Desember 2024)
# Sesuaikan dengan tahun data Anda
indonesia_holidays_2024 = [
    '2024-07-17',  # Waisak
    '2024-08-17',  # Kemerdekaan RI
    '2024-09-16',  # Maulid Nabi Muhammad SAW
    '2024-12-25',  # Natal
    '2024-12-26',  # Cuti Bersama Natal
]

indonesia_holidays_2025 = [
    '2025-07-01',  # Pancasila
    '2025-08-17',  # Kemerdekaan RI
    '2025-09-05',  # Maulid Nabi Muhammad SAW (prediksi)
    '2025-12-25',  # Natal
    '2025-12-26',  # Cuti Bersama Natal
]

# Gabungkan semua holidays dan convert ke datetime
all_holidays = pd.to_datetime(indonesia_holidays_2024 + indonesia_holidays_2025)
df_fe['is_holiday'] = df_fe['transaction_timestamp'].dt.date.isin(all_holidays.date).astype(int)

# c. Is weekend
df_fe['is_weekend'] = (df_fe['day_of_week'] >= 5).astype(int)  # Saturday=5, Sunday=6

# d. Time bins
df_fe['time_category'] = pd.cut(
    df_fe['hour'],
    bins=[0, 6, 12, 18, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening'],
    include_lowest=True
)

# e. Business hours (08:00 - 17:00)
df_fe['is_business_hours'] = ((df_fe['hour'] >= 8) & (df_fe['hour'] < 17)).astype(int)

# f. IMPROVED: Extreme late night only (00:00 - 04:00) - truly suspicious
# Hapus is_late_night yang terlalu general (22:00-05:00)
# Jam 22:00-23:59 masih wajar untuk banyak bisnis
df_fe['is_extreme_late_night'] = ((df_fe['hour'] >= 0) & (df_fe['hour'] < 4)).astype(int)

# g. Peak hours (11:00-14:00 dan 18:00-20:00)
df_fe['is_peak_hours'] = (
    ((df_fe['hour'] >= 11) & (df_fe['hour'] < 14)) |
    ((df_fe['hour'] >= 18) & (df_fe['hour'] < 20))
).astype(int)

print("✅ Time-based features created!")
print(f"\nTime feature columns added:")
time_features = ['hour', 'day', 'month', 'year', 'day_of_week', 'day_of_year', 
                 'is_holiday', 'is_weekend', 'time_category', 'is_business_hours', 
                 'is_extreme_late_night', 'is_peak_hours']
for feat in time_features:
    print(f"  - {feat}")

## 2.2 Check Time-Based Features Distribution

In [ ]:
# Check time-based features distribution
print("=== Time-based Features Distribution ===")
print(f"\nHolidays: {df_fe['is_holiday'].sum():,} transaksi ({df_fe['is_holiday'].mean()*100:.2f}%)")
print(f"Weekends: {df_fe['is_weekend'].sum():,} transaksi ({df_fe['is_weekend'].mean()*100:.2f}%)")
print(f"Business Hours: {df_fe['is_business_hours'].sum():,} transaksi ({df_fe['is_business_hours'].mean()*100:.2f}%)")
print(f"Extreme Late Night (00:00-04:00): {df_fe['is_extreme_late_night'].sum():,} transaksi ({df_fe['is_extreme_late_night'].mean()*100:.2f}%)")
print(f"Peak Hours: {df_fe['is_peak_hours'].sum():,} transaksi ({df_fe['is_peak_hours'].mean()*100:.2f}%)")

print("\n=== Time Category Distribution ===")
print(df_fe['time_category'].value_counts().sort_index())

## 2.3 Transaction amount pattern analysis for anomaly detection

In [ ]:
# a. Log transformation untuk amount (handle skewness)
df_fe['amount_log'] = np.log1p(df_fe['amount'])  # log1p untuk handle nilai 0

# b. Amount categories
df_fe['amount_category'] = pd.cut(
    df_fe['amount'],
    bins=[0, 1000, 100000, 2000000, 6500000, float('inf')],
    labels=['Micro', 'Small', 'Medium', 'Large', 'Very Large']
)

# c. IMPROVED: Hanya deteksi limit-gaming pattern (mencoba hindari limit 10M)
# Nominal bulat seperti 100rb, 500rb, 1jt, 4.25jt itu NORMAL - BUKAN anomaly!
df_fe['is_limit_gaming'] = (
    (df_fe['amount'] == 9999999) |  # Tepat di bawah 10M
    (df_fe['amount'] == 9999990) |  # Tepat di bawah 10M
    (df_fe['amount'] == 9999900)    # Tepat di bawah 10M
).astype(int)

# d. Unusual amount patterns
df_fe['is_micro_transaction'] = (df_fe['amount'] <= 1000).astype(int)

# IMPROVED: Threshold high value dinaikkan
df_fe['is_very_high_value'] = (df_fe['amount'] >= 50000000).astype(int)  # >= 50 juta baru suspicious

# e. Transaksi maksimal harian
df_fe['exceeds_daily_limit'] = (df_fe['amount'] > 10000000).astype(int)

# f. Transaksi yang mendekati maksimal (dalam range 9.9M - 10M)
df_fe['is_near_max_limit'] = (
    (df_fe['amount'] >= 9900000) & (df_fe['amount'] <= 10000000)
).astype(int)

print("✅ Basic amount features created!")
print(f"   - Limit-gaming patterns: {df_fe['is_limit_gaming'].sum():,}")
print(f"   - Very high value (>=50M): {df_fe['is_very_high_value'].sum():,}")
print(f"   - Near max limit (9.9M-10M): {df_fe['is_near_max_limit'].sum():,}")

## 2.4 Analysis of Transaction Frequency and Velocity for Abnormal Activity Detection

In [ ]:
# Create date and hour identifiers
df_fe['date'] = df_fe['transaction_timestamp'].dt.date
df_fe['date_hour'] = df_fe['transaction_timestamp'].dt.floor('H')

# Transaction count per per day
daily_counts = df_fe.groupby(['id', 'date']).size().reset_index(name='trx_count')
df_fe = df_fe.merge(daily_counts, on=['id', 'date'], how='left')

print("✅ Transaction count features created!")

## 2.5 Analysis of Transaction Characteristics and Behavior

In [ ]:
# a. Total transaction
total_trx = df_fe.groupby('id').size().reset_index(name='total_trx')
df_fe = df_fe.merge(total_trx, on='id', how='left')

print("✅ Basic features created!")

# Assume we have transaction_status column (SUCCESS, FAILED, REFUNDED)
# Adjust column names based on your actual data
if 'status_name' in df_fe.columns:
    # Success rate
    success = df_fe.groupby('id')['status_name'].apply(
        lambda x: (x == 'SUCCESS').sum() / len(x) if len(x) > 0 else 0
    ).reset_index(name='success_rate')
    df_fe = df_fe.merge(success, on='id', how='left')
    
    # Refund rate
    refund = df_fe.groupby('id')['status_name'].apply(
        lambda x: (x == 'REFUNDED').sum() / len(x) if len(x) > 0 else 0
    ).reset_index(name='refund_rate')
    df_fe = df_fe.merge(refund, on='id', how='left')
    
    print("✅ Success and refund rates created!")
else:
    print("⚠️ transaction_status column not found. Skipping success/refund rates.")

## 2.6 Historical Statistical and Trend Analysis Using Rolling Windows

In [ ]:
# a. Rolling windows: last 3, 5, 10, 50 transactions
windows = [3, 5, 10, 50]

for window in windows:
    # Rolling mean amount
    df_fe[f'rolling_mean_{window}'] = (
        df_fe.groupby('id')['amount']
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )
    
    # Rolling std amount
    df_fe[f'rolling_std_{window}'] = (
        df_fe.groupby('id')['amount']
        .rolling(window=window, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    )
    
    # Rolling max amount
    df_fe[f'rolling_max_{window}'] = (
        df_fe.groupby('id')['amount']
        .rolling(window=window, min_periods=1)
        .max()
        .reset_index(level=0, drop=True)
    )
    
    # Rolling min amount
    df_fe[f'rolling_min_{window}'] = (
        df_fe.groupby('id')['amount']
        .rolling(window=window, min_periods=1)
        .min()
        .reset_index(level=0, drop=True)
    )
    
    print(f"✅ Rolling features with window={window} created!")

In [ ]:
# b. Amount deviation from rolling mean
for window in [3, 5, 10]:
    df_fe[f'amount_dev_from_rolling_mean_{window}'] = (
        df_fe['amount'] - df_fe[f'rolling_mean_{window}']
    )
    
    df_fe[f'amount_dev_from_rolling_mean_{window}_pct'] = (
        (df_fe['amount'] - df_fe[f'rolling_mean_{window}']) / 
        df_fe[f'rolling_mean_{window}'].replace(0, 1) * 100
    )

print("✅ Rolling deviation features created!")

In [ ]:
# c. Trend indicator: is amount increasing?
# Compare current amount with rolling mean of last 5 transactions
df_fe['is_amount_increasing'] = (df_fe['amount'] > df_fe['rolling_mean_5']).astype(int)

# Trend: compare last 3 vs previous 3 transactions
df_fe['prev_amount'] = df_fe.groupby('id')['amount'].shift(1)
df_fe['is_amount_higher_than_prev'] = (df_fe['amount'] > df_fe['prev_amount'].fillna(0)).astype(int)

print("✅ Trend indicators created!")
print("\n=== Rolling Features Summary ===")
print(f"Transactions with increasing amounts: {df_fe['is_amount_increasing'].sum():,} ({df_fe['is_amount_increasing'].mean()*100:.2f}%)")

## 2.7 Categorical Encoding & Pattern Detection

In [ ]:
# a. Same amount pattern (transaksi dengan nominal yang sama berulang dalam beberapa menit terakhir)
# Count same amounts in last 5 minutes
df_fe['time_5min_bin'] = df_fe['transaction_timestamp'].dt.floor('5min')

same_amount_counts = df_fe.groupby(['id', 'time_5min_bin', 'amount']).size().reset_index(name='same_amount_5min_count')
df_fe = df_fe.merge(same_amount_counts, on=['id', 'time_5min_bin', 'amount'], how='left')

df_fe['has_same_amount_pattern'] = (df_fe['same_amount_5min_count'] >= 3).astype(int)

print("✅ Same amount pattern detection created!")

In [ ]:
# b. Consecutive same amount (3+ transaksi berurutan dengan nominal sama)
df_fe['prev_amount_1'] = df_fe.groupby('id')['amount'].shift(1)
df_fe['prev_amount_2'] = df_fe.groupby('id')['amount'].shift(2)

df_fe['consecutive_same_amount'] = (
    (df_fe['amount'] == df_fe['prev_amount_1']) & 
    (df_fe['amount'] == df_fe['prev_amount_2'])
).astype(int)

print("✅ Consecutive same amount detection created!")

In [ ]:
# c. Same hour pattern
hour_counts = df_fe.groupby(['id', 'id', 'date_hour']).size().reset_index(name='hour_trx_count')
df_fe = df_fe.merge(hour_counts, on=['id', 'id', 'date_hour'], how='left')

# Same pattern within 5 minutes
df_fe['time_5min_bin'] = df_fe['transaction_date'].dt.floor('5min')
5min_counts = df_fe.groupby(['id', 'id', 'time_5min_bin']).size().reset_index(name='5min_trx_count')
df_fe = df_fe.merge(5min_counts, on=['id', 'id', 'time_5min_bin'], how='left')

df_fe['is_high_freq_5min'] = (df_fe['5min_trx_count'] >= 5).astype(int)

print("✅ Time pattern detection created!")

In [ ]:
# d. Dynamic with very high frequency is suspicious
if 'type' in df_fe.columns:
    df_fe['is_suspicious_dynamic'] = (
        (df_fe['type'] == 'DYNAMIC') & 
        (df_fe['hourly_trx_count'] >= 20)
    ).astype(int)
    
    # e. Static with very high value is suspicious
    df_fe['is_suspicious_static'] = (
        (df_fe['type'] == 'STATIC') & 
        (df_fe['amount'] >= 9000000)
    ).astype(int)
    
    print("✅ Suspicious patterns created!")
else:
    print("⚠️ type column not found. Skipping pattern detection.")

In [ ]:
# f. Check if has multiple transactions in same hour
hour_counts = df_fe.groupby(['id', 'date_hour'])['id'].nunique().reset_index(name='hour_unique')
df_fe = df_fe.merge(hour_counts, on=['id', 'date_hour'], how='left')

df_fe['has_multi_same_hour'] = (df_fe['hour_unique'] > 1).astype(int)

print("✅ Multiple detection created!")

In [ ]:
# g. Failed transaction followed by success (retry pattern)
if 'status_name' in df_fe.columns:
    df_fe['prev_status'] = df_fe.groupby('id')['status_name'].shift(1)
    
    df_fe['is_retry_pattern'] = (
        (df_fe['prev_status'] == 'FAILED') & 
        (df_fe['status_name'] == 'SUCCESS')
    ).astype(int)
    
    # h. Refund pattern
    df_fe['is_refund'] = (df_fe['status_name'] == 'REFUNDED').astype(int)
    
    # Count refunds in last hour
    refund_hourly = df_fe[df_fe['status_name'] == 'REFUNDED'].groupby(
        ['id', 'date_hour']
    ).size().reset_index(name='hourly_refund_count')
    df_fe = df_fe.merge(refund_hourly, on=['id', 'date_hour'], how='left')
    df_fe['hourly_refund_count'].fillna(0, inplace=True)
    
    df_fe['has_high_refund_pattern'] = (df_fe['hourly_refund_count'] >= 3).astype(int)
    
    print("✅ Retry and refund patterns created!")
else:
    print("⚠️ transaction_status column not found. Skipping retry/refund patterns.")

print("\n=== Pattern Detection Summary ===")
print(f"Same amount patterns (3+ in 5min): {df_fe['has_same_amount_pattern'].sum():,} ({df_fe['has_same_amount_pattern'].mean()*100:.2f}%)")
print(f"Consecutive same amounts: {df_fe['consecutive_same_amount'].sum():,} ({df_fe['consecutive_same_amount'].mean()*100:.2f}%)")
print(f"High freq (5+ in 5min): {df_fe['is_high_freq_5min'].sum():,} ({df_fe['is_high_freq_5min'].mean()*100:.2f}%)")

# 3. Anomaly Detection with Isolation Forest

## 3.1 Data Preparation & Feature Selection

In [ ]:
# Identifikasi tipe kolom
print("=== Column Types Analysis ===\n")

# ID columns (tidak digunakan untuk modeling)
id_cols = ['transaction_id', 'id']

# Datetime columns (tidak digunakan langsung)
datetime_cols = ['transaction_date', 'transaction_timestamp']

# Categorical columns
categorical_cols = []

# Numeric columns (akan di-scale)
numeric_cols = [col for col in df.columns 
                if col not in id_cols + datetime_cols + categorical_cols 
                and df[col].dtype in ['int64', 'float64']]

print(f"ID columns: {len(id_cols)}")
print(f"Datetime columns: {len(datetime_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

print(f"\n=== Categorical Columns ===")
for col in categorical_cols:
    if col in df.columns:
        n_unique = df[col].nunique()
        print(f"{col}: {n_unique} unique values")

## 3.2 Encoding

In [ ]:
# Kategorikan berdasarkan jumlah unique values
small_categorical = []  # untuk one-hot encoding
large_categorical = []  # untuk frequency encoding

threshold_unique = 10  # threshold untuk memisahkan

for col in categorical_cols:
    if col in df_model.columns:
        n_unique = df_model[col].nunique()
        if n_unique < threshold_unique:
            small_categorical.append(col)
        else:
            large_categorical.append(col)

print("=== Encoding Strategy ===\n")
print(f"One-Hot Encoding (< {threshold_unique} categories):")
for col in small_categorical:
    print(f"  - {col}: {df_model[col].nunique()} categories")

print(f"\nFrequency Encoding (>= {threshold_unique} categories):")
for col in large_categorical:
    print(f"  - {col}: {df_model[col].nunique()} categories")

In [ ]:
# 1. One-Hot Encoding untuk kategori kecil
print("=== Applying One-Hot Encoding ===\n")

df_encoded = df_model.copy()

for col in small_categorical:
    if col in df_encoded.columns:
        # One-hot encoding
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        df_encoded.drop(columns=[col], inplace=True)
        print(f"✅ {col}: created {len(dummies.columns)} dummy variables")

print(f"\nDataset shape after one-hot encoding: {df_encoded.shape}")

In [ ]:
# 2. Frequency Encoding untuk kategori besar
print("=== Applying Frequency Encoding ===\n")

for col in large_categorical:
    if col in df_encoded.columns:
        # Hitung frekuensi setiap kategori
        freq_encoding = df_encoded[col].value_counts(normalize=True).to_dict()
        
        # Apply frequency encoding
        df_encoded[f'{col}_freq'] = df_encoded[col].map(freq_encoding)
        
        # Drop original column
        df_encoded.drop(columns=[col], inplace=True)
        
        print(f"✅ {col}: frequency encoded (range: {df_encoded[f'{col}_freq'].min():.4f} - {df_encoded[f'{col}_freq'].max():.4f})")

print(f"\nDataset shape after frequency encoding: {df_encoded.shape}")

## 3.3 Feature Selection & Scaling

In [ ]:
# Pisahkan fitur untuk modeling dan metadata
metadata_cols = []

# Ambil semua kolom numerik (excluding metadata)
feature_cols = [col for col in df_encoded.columns 
                if col not in metadata_cols 
                and df_encoded[col].dtype in ['int64', 'float64', 'uint8']]

print(f"=== Feature Selection ===\n")
print(f"Total features for modeling: {len(feature_cols)}")
print(f"Metadata columns (excluded): {len(metadata_cols)}")

# Create feature matrix
X = df_encoded[feature_cols].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Handle missing values (fill with 0 or median)
print("=== Handling Missing Values ===\n")

# Check missing values per column
missing_summary = X.isnull().sum()
missing_cols = missing_summary[missing_summary > 0]

if len(missing_cols) > 0:
    print(f"Columns with missing values: {len(missing_cols)}")
    for col in missing_cols.index[:10]:  # show top 10
        print(f"  - {col}: {missing_cols[col]:,} ({missing_cols[col]/len(X)*100:.2f}%)")
    
    # Fill missing values with 0 (most features are indicators/counts)
    X.fillna(0, inplace=True)
    print(f"\n✅ Missing values filled with 0")
else:
    print("✅ No missing values found")

In [ ]:
# Apply RobustScaler (robust terhadap outliers)
print("=== Applying RobustScaler ===\n")

scaler = RobustScaler()

# Fit and transform
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"✅ Features scaled with RobustScaler")
print(f"\nScaled data range:")
print(f"  Min: {X_scaled.min().min():.4f}")
print(f"  Max: {X_scaled.max().max():.4f}")
print(f"  Mean: {X_scaled.mean().mean():.4f}")
print(f"  Std: {X_scaled.std().mean():.4f}")

In [ ]:
# Visualisasi distribusi sebelum dan sesudah scaling (sample beberapa fitur)
sample_features = X.columns[:6]  # 6 fitur pertama

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, col in enumerate(sample_features):
    axes[idx].hist(X_scaled[col], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{col}', fontsize=10)
    axes[idx].set_xlabel('Scaled Value')
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('feature_distribution_scaled.png', dpi=150, bbox_inches='tight')
print("\n✅ Distribution plot saved: feature_distribution_scaled.png")
plt.show()

## 3.4 Train/Test Split

In [ ]:
# Train/Test Split (80-20)
print("=== Train/Test Split ===\n")

# Gunakan random_state untuk reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, idx_train, idx_test = train_test_split(
    X_scaled, 
    X_scaled.index, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Training set: {X_train.shape[0]:,} samples ({(1-TEST_SIZE)*100:.0f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({TEST_SIZE*100:.0f}%)")
print(f"Features: {X_train.shape[1]}")

# Simpan metadata untuk train dan test
train_metadata = df_encoded.loc[idx_train, metadata_cols].copy()
test_metadata = df_encoded.loc[idx_test, metadata_cols].copy()

print(f"\n✅ Train/test split completed")

## 3.5 Isolation Forest - Baseline Model

In [ ]:
# Baseline Isolation Forest
print("=== Training Baseline Isolation Forest ===\n")

# IMPROVED: Parameters baseline - contamination dikurangi dari 0.05 ke 0.02
# 5% terlalu banyak untuk fraud detection, lebih realistic 2% yang truly anomalous
contamination = 0.02  # Turunkan dari 0.05 ke 0.02 (2% lebih realistic)

# Initialize model
iso_forest_baseline = IsolationForest(
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1,  # use all CPU cores
    verbose=0
)

# Fit model
print("Training baseline model...")
start_time = time.time()
iso_forest_baseline.fit(X_train)
training_time = time.time() - start_time

print(f"✅ Baseline model trained in {training_time:.2f} seconds")
print(f"\nModel parameters:")
print(f"  - n_estimators: {iso_forest_baseline.n_estimators}")
print(f"  - max_samples: {iso_forest_baseline.max_samples}")
print(f"  - contamination: {contamination}")
print(f"  - random_state: {RANDOM_STATE}")

## 3.6 Scoring & Prediction

In [ ]:
# Predict anomaly scores
print("=== Generating Anomaly Scores ===\n")

# Anomaly scores (semakin negatif = semakin anomali)
train_scores = iso_forest_baseline.score_samples(X_train)
test_scores = iso_forest_baseline.score_samples(X_test)

# Predictions (-1 = anomaly, 1 = normal)
train_predictions = iso_forest_baseline.predict(X_train)
test_predictions = iso_forest_baseline.predict(X_test)

# Invert scores (semakin besar = semakin anomali)
train_anomaly_scores = -train_scores
test_anomaly_scores = -test_scores

print("✅ Anomaly scores generated")
print(f"\nTrain scores range: [{train_anomaly_scores.min():.4f}, {train_anomaly_scores.max():.4f}]")
print(f"Test scores range: [{test_anomaly_scores.min():.4f}, {test_anomaly_scores.max():.4f}]")

print(f"\nTrain anomalies detected: {(train_predictions == -1).sum():,} ({(train_predictions == -1).mean()*100:.2f}%)")
print(f"Test anomalies detected: {(test_predictions == -1).sum():,} ({(test_predictions == -1).mean()*100:.2f}%)")

In [ ]:
# Tambahkan scores ke metadata
train_metadata['anomaly_score'] = train_anomaly_scores
train_metadata['is_anomaly'] = (train_predictions == -1).astype(int)
train_metadata['split'] = 'train'

test_metadata['anomaly_score'] = test_anomaly_scores
test_metadata['is_anomaly'] = (test_predictions == -1).astype(int)
test_metadata['split'] = 'test'

# Gabungkan train dan test metadata
results_df = pd.concat([train_metadata, test_metadata], axis=0).sort_index()

# Merge dengan data original (df) untuk mendapatkan informasi tambahan termasuk status_name
results_df = results_df.merge(
    df[['amount', 'status_name']],
    left_index=True,
    right_index=True,
    how='left'
)

print(f"✅ Results dataframe created: {results_df.shape}")

In [ ]:
# Visualisasi distribusi anomaly scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Train scores
axes[0].hist(train_anomaly_scores, bins=100, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(np.percentile(train_anomaly_scores, 95), color='orange', linestyle='--', 
                label='95th percentile', linewidth=2)
axes[0].axvline(np.percentile(train_anomaly_scores, 99), color='red', linestyle='--', 
                label='99th percentile', linewidth=2)
axes[0].set_title('Train Set - Anomaly Score Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test scores
axes[1].hist(test_anomaly_scores, bins=100, alpha=0.7, color='coral', edgecolor='black')
axes[1].axvline(np.percentile(test_anomaly_scores, 95), color='orange', linestyle='--', 
                label='95th percentile', linewidth=2)
axes[1].axvline(np.percentile(test_anomaly_scores, 99), color='red', linestyle='--', 
                label='99th percentile', linewidth=2)
axes[1].set_title('Test Set - Anomaly Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
print("\n✅ Anomaly score distribution saved: anomaly_score_distribution.png")
plt.show()

## 3.7 Thresholding

In [ ]:
# Calculate thresholds berdasarkan training data
print("=== Calculating Thresholds ===\n")

# 95th percentile - Suspicious
threshold_suspicious = np.percentile(train_anomaly_scores, 95)

# 99th percentile - High Risk
threshold_high_risk = np.percentile(train_anomaly_scores, 99)

print(f"Threshold Suspicious (95th percentile): {threshold_suspicious:.4f}")
print(f"Threshold High Risk (99th percentile): {threshold_high_risk:.4f}")

# Apply thresholds ke seluruh dataset
results_df['risk_level'] = 'normal'
results_df.loc[results_df['anomaly_score'] >= threshold_suspicious, 'risk_level'] = 'suspicious'
results_df.loc[results_df['anomaly_score'] >= threshold_high_risk, 'risk_level'] = 'high_risk'

# Summary
print(f"\n=== Risk Level Distribution ===")
print(results_df['risk_level'].value_counts())
print(f"\n{results_df['risk_level'].value_counts(normalize=True) * 100}")

In [ ]:
# Visualisasi risk levels
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Count plot
risk_counts = results_df['risk_level'].value_counts()
colors = {'normal': 'green', 'suspicious': 'orange', 'high_risk': 'red'}
risk_colors = [colors[level] for level in risk_counts.index]

axes[0].bar(risk_counts.index, risk_counts.values, color=risk_colors, alpha=0.7, edgecolor='black')
axes[0].set_title('Risk Level Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3, axis='y')

# Add count labels
for i, (level, count) in enumerate(risk_counts.items()):
    axes[0].text(i, count + 5000, f'{count:,}\n({count/len(results_df)*100:.1f}%)', 
                ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', 
           colors=risk_colors, startangle=90, explode=(0.05, 0.05, 0.1))
axes[1].set_title('Risk Level Proportion', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('risk_level_distribution.png', dpi=150, bbox_inches='tight')
print("\n✅ Risk level distribution saved: risk_level_distribution.png")
plt.show()

## 3.8 Model Evaluation - Overfitting Check

In [ ]:
# Evaluasi performa train vs test
print("=== Model Performance Evaluation ===\n")

# Train metrics
train_normal = (train_predictions == 1).sum()
train_anomaly = (train_predictions == -1).sum()
train_anomaly_pct = train_anomaly / len(train_predictions) * 100

# Test metrics
test_normal = (test_predictions == 1).sum()
test_anomaly = (test_predictions == -1).sum()
test_anomaly_pct = test_anomaly / len(test_predictions) * 100

print("Train Set:")
print(f"  Normal: {train_normal:,} ({100-train_anomaly_pct:.2f}%)")
print(f"  Anomaly: {train_anomaly:,} ({train_anomaly_pct:.2f}%)")

print("\nTest Set:")
print(f"  Normal: {test_normal:,} ({100-test_anomaly_pct:.2f}%)")
print(f"  Anomaly: {test_anomaly:,} ({test_anomaly_pct:.2f}%)")

# Check overfitting: anomaly rate seharusnya mirip antara train dan test
diff_pct = abs(train_anomaly_pct - test_anomaly_pct)
print(f"\nAnomaly Rate Difference: {diff_pct:.2f}%")

if diff_pct < 1.0:
    print("✅ Model is WELL-GENERALIZED (no overfitting)")
elif diff_pct < 3.0:
    print("⚠️ Model shows SLIGHT variance (acceptable)")
else:
    print("❌ Model may be OVERFITTING (significant difference)")

In [ ]:
# Score statistics comparison
print("\n=== Anomaly Score Statistics ===\n")

stats_df = pd.DataFrame({
    'Train': [
        train_anomaly_scores.mean(),
        train_anomaly_scores.std(),
        train_anomaly_scores.min(),
        np.percentile(train_anomaly_scores, 25),
        np.percentile(train_anomaly_scores, 50),
        np.percentile(train_anomaly_scores, 75),
        train_anomaly_scores.max()
    ],
    'Test': [
        test_anomaly_scores.mean(),
        test_anomaly_scores.std(),
        test_anomaly_scores.min(),
        np.percentile(test_anomaly_scores, 25),
        np.percentile(test_anomaly_scores, 50),
        np.percentile(test_anomaly_scores, 75),
        test_anomaly_scores.max()
    ]
}, index=['Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max'])

print(stats_df.round(4))

# Check similarity
mean_diff = abs(stats_df.loc['Mean', 'Train'] - stats_df.loc['Mean', 'Test'])
print(f"\nMean difference: {mean_diff:.4f}")

if mean_diff < 0.1:
    print("✅ Score distributions are VERY SIMILAR")
elif mean_diff < 0.2:
    print("✅ Score distributions are SIMILAR")
else:
    print("⚠️ Score distributions show some DIFFERENCE")

## 3.9 Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning - Stage 1: n_estimators
print("=== Hyperparameter Tuning - Stage 1: n_estimators ===\n")

n_estimators_range = [300, 500, 700, 1000]
results_stage1 = []

for n_est in n_estimators_range:
    print(f"Testing n_estimators={n_est}...")
    
    model = IsolationForest(
        n_estimators=n_est,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Train
    start_time = time.time()
    model.fit(X_train)
    train_time = time.time() - start_time
    
    # Evaluate
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_anomaly_rate = (train_pred == -1).mean()
    test_anomaly_rate = (test_pred == -1).mean()
    
    # Calculate scores
    train_scores_temp = -model.score_samples(X_train)
    test_scores_temp = -model.score_samples(X_test)
    
    results_stage1.append({
        'n_estimators': n_est,
        'train_time': train_time,
        'train_anomaly_rate': train_anomaly_rate,
        'test_anomaly_rate': test_anomaly_rate,
        'score_diff': abs(train_anomaly_rate - test_anomaly_rate),
        'train_score_mean': train_scores_temp.mean(),
        'test_score_mean': test_scores_temp.mean()
    })
    
    print(f"  Train time: {train_time:.2f}s | Train anomaly: {train_anomaly_rate*100:.2f}% | Test anomaly: {test_anomaly_rate*100:.2f}%\n")

# Convert to DataFrame
results_stage1_df = pd.DataFrame(results_stage1)
print("\n=== Stage 1 Results ===")
print(results_stage1_df)

# Best n_estimators (lowest score_diff)
best_n_estimators = results_stage1_df.loc[results_stage1_df['score_diff'].idxmin(), 'n_estimators']
print(f"\n✅ Best n_estimators: {int(best_n_estimators)}")

In [ ]:
# Hyperparameter tuning - Stage 2: max_samples
print("=== Hyperparameter Tuning - Stage 2: max_samples ===\n")

max_samples_range = [0.7, 0.9]
results_stage2 = []

for max_samp in max_samples_range:
    print(f"Testing max_samples={max_samp}...")
    
    model = IsolationForest(
        n_estimators=int(best_n_estimators),
        max_samples=max_samp,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Train
    start_time = time.time()
    model.fit(X_train)
    train_time = time.time() - start_time
    
    # Evaluate
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_anomaly_rate = (train_pred == -1).mean()
    test_anomaly_rate = (test_pred == -1).mean()
    
    # Calculate scores
    train_scores_temp = -model.score_samples(X_train)
    test_scores_temp = -model.score_samples(X_test)
    
    results_stage2.append({
        'max_samples': max_samp,
        'train_time': train_time,
        'train_anomaly_rate': train_anomaly_rate,
        'test_anomaly_rate': test_anomaly_rate,
        'score_diff': abs(train_anomaly_rate - test_anomaly_rate),
        'train_score_mean': train_scores_temp.mean(),
        'test_score_mean': test_scores_temp.mean()
    })
    
    print(f"  Train time: {train_time:.2f}s | Train anomaly: {train_anomaly_rate*100:.2f}% | Test anomaly: {test_anomaly_rate*100:.2f}%\n")

# Convert to DataFrame
results_stage2_df = pd.DataFrame(results_stage2)
print("\n=== Stage 2 Results ===")
print(results_stage2_df)

# Best max_samples
best_max_samples = results_stage2_df.loc[results_stage2_df['score_diff'].idxmin(), 'max_samples']
print(f"\n✅ Best max_samples: {best_max_samples}")

In [ ]:
# Hyperparameter tuning - Stage 3: max_features
print("=== Hyperparameter Tuning - Stage 3: max_features ===\n")

max_features_range = [1.0]
results_stage3 = []

for max_feat in max_features_range:
    print(f"Testing max_features={max_feat}...")
    
    model = IsolationForest(
        n_estimators=int(best_n_estimators),
        max_samples=best_max_samples,
        max_features=max_feat,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    # Train
    start_time = time.time()
    model.fit(X_train)
    train_time = time.time() - start_time
    
    # Evaluate
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    train_anomaly_rate = (train_pred == -1).mean()
    test_anomaly_rate = (test_pred == -1).mean()
    
    # Calculate scores
    train_scores_temp = -model.score_samples(X_train)
    test_scores_temp = -model.score_samples(X_test)
    
    results_stage3.append({
        'max_features': max_feat,
        'train_time': train_time,
        'train_anomaly_rate': train_anomaly_rate,
        'test_anomaly_rate': test_anomaly_rate,
        'score_diff': abs(train_anomaly_rate - test_anomaly_rate),
        'train_score_mean': train_scores_temp.mean(),
        'test_score_mean': test_scores_temp.mean()
    })
    
    print(f"  Train time: {train_time:.2f}s | Train anomaly: {train_anomaly_rate*100:.2f}% | Test anomaly: {test_anomaly_rate*100:.2f}%\n")

# Convert to DataFrame
results_stage3_df = pd.DataFrame(results_stage3)
print("\n=== Stage 3 Results ===")
print(results_stage3_df)

# Best max_features
best_max_features = results_stage3_df.loc[results_stage3_df['score_diff'].idxmin(), 'max_features']
print(f"\n✅ Best max_features: {best_max_features}")

In [ ]:
# Summary of best parameters
print("=" * 80)
print("BEST HYPERPARAMETERS")
print("=" * 80)
print(f"\nn_estimators: {int(best_n_estimators)}")
print(f"max_samples: {best_max_samples}")
print(f"max_features: {best_max_features}")
print(f"contamination: {contamination}")
print(f"random_state: {RANDOM_STATE}")

# Visualisasi tuning results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage 1
axes[0].plot(results_stage1_df['n_estimators'], results_stage1_df['score_diff'], 
            marker='o', linewidth=2, markersize=8)
axes[0].set_title('Stage 1: n_estimators', fontsize=12, fontweight='bold')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Score Difference')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(best_n_estimators, color='red', linestyle='--', label='Best')
axes[0].legend()

# Stage 2
axes[1].plot(results_stage2_df['max_samples'], results_stage2_df['score_diff'], 
            marker='s', linewidth=2, markersize=8, color='orange')
axes[1].set_title('Stage 2: max_samples', fontsize=12, fontweight='bold')
axes[1].set_xlabel('max_samples')
axes[1].set_ylabel('Score Difference')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(best_max_samples, color='red', linestyle='--', label='Best')
axes[1].legend()

# Stage 3
axes[2].plot(results_stage3_df['max_features'], results_stage3_df['score_diff'], 
            marker='^', linewidth=2, markersize=8, color='green')
axes[2].set_title('Stage 3: max_features', fontsize=12, fontweight='bold')
axes[2].set_xlabel('max_features')
axes[2].set_ylabel('Score Difference')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(best_max_features, color='red', linestyle='--', label='Best')
axes[2].legend()

plt.tight_layout()
plt.savefig('hyperparameter_tuning_results.png', dpi=150, bbox_inches='tight')
print("\n✅ Tuning results saved: hyperparameter_tuning_results.png")
plt.show()

## 3.10 Final Model with Best Parameters

In [ ]:
# Train final model dengan best parameters
print("=== Training Final Model with Best Parameters ===\n")

final_model = IsolationForest(
    n_estimators=int(best_n_estimators),
    max_samples=best_max_samples,
    max_features=best_max_features,
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

# Fit on entire training data
print("Training final model...")
start_time = time.time()
final_model.fit(X_train)
final_train_time = time.time() - start_time

print(f"\n✅ Final model trained in {final_train_time:.2f} seconds")
print(f"\nFinal Model Parameters:")
print(f"  - n_estimators: {final_model.n_estimators}")
print(f"  - max_samples: {final_model.max_samples}")
print(f"  - max_features: {final_model.max_features}")
print(f"  - contamination: {final_model.contamination}")

In [ ]:
# Generate final predictions dan scores
print("=== Generating Final Predictions ===\n")

# Predict on train and test
final_train_pred = final_model.predict(X_train)
final_test_pred = final_model.predict(X_test)

# Anomaly scores
final_train_scores = -final_model.score_samples(X_train)
final_test_scores = -final_model.score_samples(X_test)

print(f"Train anomalies: {(final_train_pred == -1).sum():,} ({(final_train_pred == -1).mean()*100:.2f}%)")
print(f"Test anomalies: {(final_test_pred == -1).sum():,} ({(final_test_pred == -1).mean()*100:.2f}%)")

# Update results_df dengan final scores
results_df.loc[idx_train, 'final_anomaly_score'] = final_train_scores
results_df.loc[idx_test, 'final_anomaly_score'] = final_test_scores
results_df.loc[idx_train, 'final_is_anomaly'] = (final_train_pred == -1).astype(int)
results_df.loc[idx_test, 'final_is_anomaly'] = (final_test_pred == -1).astype(int)

# Recalculate thresholds dengan final model
final_threshold_suspicious = np.percentile(final_train_scores, 95)
final_threshold_high_risk = np.percentile(final_train_scores, 99)

print(f"\nFinal Thresholds:")
print(f"  Suspicious (95th): {final_threshold_suspicious:.4f}")
print(f"  High Risk (99th): {final_threshold_high_risk:.4f}")

# Apply final risk levels
results_df['final_risk_level'] = 'normal'
results_df.loc[results_df['final_anomaly_score'] >= final_threshold_suspicious, 'final_risk_level'] = 'suspicious'
results_df.loc[results_df['final_anomaly_score'] >= final_threshold_high_risk, 'final_risk_level'] = 'high_risk'

print(f"\n✅ Final predictions and risk levels assigned")

In [ ]:
# Final model evaluation
print("=== Final Model Evaluation ===\n")

# Compare baseline vs final
comparison_df = pd.DataFrame({
    'Baseline Train': [
        (train_predictions == -1).mean() * 100,
        train_anomaly_scores.mean(),
        train_anomaly_scores.std()
    ],
    'Baseline Test': [
        (test_predictions == -1).mean() * 100,
        test_anomaly_scores.mean(),
        test_anomaly_scores.std()
    ],
    'Final Train': [
        (final_train_pred == -1).mean() * 100,
        final_train_scores.mean(),
        final_train_scores.std()
    ],
    'Final Test': [
        (final_test_pred == -1).mean() * 100,
        final_test_scores.mean(),
        final_test_scores.std()
    ]
}, index=['Anomaly Rate (%)', 'Score Mean', 'Score Std'])

print(comparison_df.round(4))

# Check improvement
baseline_diff = abs((train_predictions == -1).mean() - (test_predictions == -1).mean())
final_diff = abs((final_train_pred == -1).mean() - (final_test_pred == -1).mean())

print(f"\nBaseline train-test difference: {baseline_diff*100:.2f}%")
print(f"Final train-test difference: {final_diff*100:.2f}%")

if final_diff < baseline_diff:
    print(f"✅ Final model IMPROVED generalization by {(baseline_diff - final_diff)*100:.2f}%")
else:
    print(f"⚠️ Baseline was already well-generalized")

## 3.11 Transaction Anomaly Scores

In [ ]:
# Anomaly scores
print("=== Anomaly Scores ===\n")

# Sort by final anomaly score
scores = results_df.sort_values('final_anomaly_score', ascending=False).copy()

# Add rank
scores['anomaly_rank'] = range(1, len(scores) + 1)

print(f"Total: {len(scores):,}")
print(f"\nTop 10 Most Anomalous:")
print(transaction_scores[['id', 'final_anomaly_score', 'final_risk_level', 'anomaly_rank']].head(10))

# Statistics per risk level
print(f"\n=== Statistics by Risk Level ===")
for risk_level in ['high_risk', 'suspicious', 'normal']:
    subset = scores[scores['final_risk_level'] == risk_level]
    if len(subset) > 0:
        print(f"\n{risk_level.upper()}:")
        print(f"  Count: {len(subset):,} ({len(subset)/len(scores)*100:.2f}%)")
        print(f"  Avg Amount: Rp {subset['amount'].mean():,.0f}")
        print(f"  Median Amount: Rp {subset['amount'].median():,.0f}")
        print(f"  Avg Score: {subset['final_anomaly_score'].mean():.4f}")

# 4. Prediction New Transactions

## 4.1 Load Model

In [ ]:
# Load trained model dan artifacts
print("=== Loading Model Assets ===\n")

# Load model
model = joblib.load('isolation_forest_model.pkl')
print("✅ Isolation Forest model loaded")

# Load scaler
scaler = joblib.load('robust_scaler.pkl')
print("✅ RobustScaler loaded")

# Load feature names
with open('feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]
print(f"✅ Feature names loaded: {len(feature_names)} features")

# Load thresholds
with open('thresholds.json', 'r') as f:
    thresholds = json.load(f)
    
threshold_suspicious = thresholds['suspicious']
threshold_high_risk = thresholds['high_risk']

print(f"✅ Thresholds loaded:")
print(f"   - Suspicious: {threshold_suspicious:.4f}")
print(f"   - High Risk: {threshold_high_risk:.4f}")

print("\n=== Model Configuration ===")
print(f"n_estimators: {model.n_estimators}")
print(f"max_samples: {model.max_samples}")
print(f"contamination: {model.contamination}")

## 4.2 Prediction & Scoring

In [ ]:
# Predict anomaly scores
print("=== Predicting Anomaly Scores ===\n")

# Get predictions
predictions = model.predict(X_new_scaled)
scores = model.score_samples(X_new_scaled)

# Invert scores (higher = more anomalous)
anomaly_scores = -scores

print(f"✅ Predictions completed")
print(f"\nScore statistics:")
print(f"   Min: {anomaly_scores.min():.4f}")
print(f"   Max: {anomaly_scores.max():.4f}")
print(f"   Mean: {anomaly_scores.mean():.4f}")
print(f"   Median: {np.median(anomaly_scores):.4f}")
print(f"   Std: {anomaly_scores.std():.4f}")

print(f"\nAnomaly detection:")
anomaly_count = (predictions == -1).sum()
print(f"   Anomalies: {anomaly_count:,} ({anomaly_count/len(predictions)*100:.2f}%)")
print(f"   Normal: {(predictions == 1).sum():,} ({(predictions == 1).sum()/len(predictions)*100:.2f}%)")

In [ ]:
# Apply thresholds
print("=== Applying Risk Thresholds ===\n")

# Create results dataframe with all important columns
# Get metadata columns (including names if available)
results_cols = ['transaction_id', 'id', 
                'transaction_date', 'transaction_timestamp', 'amount']

# Add name columns if they exist
if 'name' in df_encoded.columns:
    results_cols.append('name')
if 'name' in df_encoded.columns:
    results_cols.append('name')

results_df = df_encoded[results_cols].copy()

# Add categorical columns from original data
results_df['status_name'] = df_original['status_name'].values
if 'type' in df_original.columns:
    results_df['type'] = df_original['type'].values
if 'payment_method' in df_original.columns:
    results_df['payment_method'] = df_original['payment_method'].values

# Add prediction results
results_df['anomaly_score'] = anomaly_scores
results_df['is_anomaly'] = (predictions == -1).astype(int)

# Apply risk levels
results_df['risk_level'] = 'normal'
results_df.loc[results_df['anomaly_score'] >= threshold_suspicious, 'risk_level'] = 'suspicious'
results_df.loc[results_df['anomaly_score'] >= threshold_high_risk, 'risk_level'] = 'high_risk'

# Risk distribution
risk_dist = results_df['risk_level'].value_counts()

print("Risk Level Distribution:")
for level in ['normal', 'suspicious', 'high_risk']:
    count = risk_dist.get(level, 0)
    print(f"   {level.upper()}: {count:,} ({count/len(results_df)*100:.2f}%)")

In [ ]:
# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Score distribution
axes[0].hist(anomaly_scores, bins=100, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(threshold_suspicious, color='orange', linestyle='--', 
                label='Suspicious (95th)', linewidth=2)
axes[0].axvline(threshold_high_risk, color='red', linestyle='--', 
                label='High Risk (99th)', linewidth=2)
axes[0].set_title('Anomaly Score Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Risk level pie chart
colors_risk = {'normal': 'green', 'suspicious': 'orange', 'high_risk': 'red'}
risk_colors = [colors_risk[level] for level in risk_dist.index]
axes[1].pie(risk_dist.values, labels=[l.upper() for l in risk_dist.index], 
           autopct='%1.1f%%', colors=risk_colors, startangle=90, 
           explode=(0.05, 0.05, 0.1))
axes[1].set_title('Risk Level Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('new_data_anomaly_scores.png', dpi=150, bbox_inches='tight')
print("\n✅ Score distribution saved: new_data_anomaly_scores.png")
plt.show()

## 4.3 Transaction Anomaly Score

In [ ]:
# Transaction-level scores
print("=== Transaction Anomaly Scores ===\n")

transaction_scores = results_df.sort_values('anomaly_score', ascending=False).copy()
transaction_scores['anomaly_rank'] = range(1, len(transaction_scores) + 1)

print(f"Total transactions: {len(transaction_scores):,}")
print(f"\nTop 10 Most Anomalous Transactions:")
print(transaction_scores[['transaction_id', 'id', 'amount', 
                          'anomaly_score', 'risk_level', 'anomaly_rank']].head(10))

# Save
transaction_scores.to_csv('new_data_transaction_scores.csv', index=False)
print(f"\n✅ Transaction scores saved: new_data_transaction_scores.csv")

# Visualize top anomalies
top_n = 20
top_anomalies = transaction_scores.head(top_n)

plt.figure(figsize=(12, 8))
colors = ['red' if x == 'high_risk' else 'orange' if x == 'suspicious' else 'green' 
         for x in top_anomalies['risk_level']]

plt.barh(range(top_n), top_anomalies['anomaly_score'], color=colors, alpha=0.7, edgecolor='black')
plt.yticks(range(top_n), [f"Rank {i+1}" for i in range(top_n)])
plt.xlabel('Anomaly Score', fontsize=11, fontweight='bold')
plt.ylabel('Transaction Rank', fontsize=11, fontweight='bold')
plt.title(f'Top {top_n} Most Anomalous Transactions', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig('new_data_top_anomalous_transactions.png', dpi=150, bbox_inches='tight')
print("✅ Top anomalies plot saved: new_data_top_anomalous_transactions.png")
plt.show()

## 4.4 Spike Detection

In [ ]:
# Spike detection
print("=== Spike Detection - Daily Level ===\n")

results_df['date'] = results_df['transaction_timestamp'].dt.date

daily_stats = results_df.groupby('date').agg({
    'transaction_id': 'count',
    'is_anomaly': 'sum',
    'anomaly_score': 'mean',
    'amount': 'sum'
}).reset_index()

daily_stats.columns = ['date', 'total_trx', 'anomaly_count', 'avg_anomaly_score', 'total_amount']
daily_stats['anomaly_rate'] = (daily_stats['anomaly_count'] / daily_stats['total_trx'] * 100)

# Moving averages
daily_stats['trx_ma7'] = daily_stats['total_trx'].rolling(window=7, min_periods=1).mean()
daily_stats['anomaly_rate_ma7'] = daily_stats['anomaly_rate'].rolling(window=7, min_periods=1).mean()

# Detect spikes
daily_stats['trx_spike'] = daily_stats['total_trx'] > (daily_stats['trx_ma7'] * 1.5)
daily_stats['anomaly_spike'] = daily_stats['anomaly_rate'] > (daily_stats['anomaly_rate_ma7'] * 1.5)

spike_days = daily_stats[(daily_stats['trx_spike']) | (daily_stats['anomaly_spike'])]

print(f"Total days analyzed: {len(daily_stats)}")
print(f"Days with spikes: {len(spike_days)} ({len(spike_days)/len(daily_stats)*100:.1f}%)")

if len(spike_days) > 0:
    print(f"\nSpike Days:")
    print(spike_days[['date', 'total_trx', 'anomaly_count', 'anomaly_rate', 
                      'trx_spike', 'anomaly_spike']])

# Save
daily_stats.to_csv('new_data_spike_detection.csv', index=False)
print(f"\n✅ Spike detection saved: new_data_spike_detection.csv")

## 4.5 High & Very High Risk Summary

In [ ]:
# Get all high risk and very high risk transactions
print("=== High & Very High Risk Summary ===\n")

# Merge dengan data original untuk mendapatkan semua info
high_risk_df = results_df[results_df['risk_level'].isin(['suspicious', 'high_risk'])].copy()

# Merge dengan df_original untuk get all features
high_risk_df = high_risk_df.merge(
    df_original[['transaction_id', 'id', 'amount', 
                 'type', 'payment_method', 'status_name', 'is_refund',
                 'hour', 'is_late_night', 'is_business_hours', 'is_weekend',
                 'is_very_high_value', 'exceeds_daily_limit', 'is_near_max_limit',
                 'is_very_rapid_trx', 'is_high_velocity_',
                 'has_same_amount_pattern', 'consecutive_same_amount']],
    on=['transaction_id', 'id', 'amount'],
    how='left'
)

print(f"Total high/very high risk transactions: {len(high_risk_df):,}")
print(f"  - High Risk: {(high_risk_df['risk_level'] == 'high_risk').sum():,}")
print(f"  - Suspicious: {(high_risk_df['risk_level'] == 'suspicious').sum():,}")

# 5. Real Time Prediction

## 5.1 Complete Report with Description

In [ ]:
print("\n" + "=" * 100)
print("OUTPUT 7: COMPLETE HIGH RISK REPORT WITH EXPLANATIONS")
print("=" * 100)

if len(high_very_high) > 0:
    # Ensure high_very_high has no duplicates
    high_very_high_clean = high_very_high.drop_duplicates(subset=['transaction_id'], keep='first')
    
    # Ensure df_original has no duplicates before merge
    df_original_clean = df_original.drop_duplicates(subset=['transaction_id', 'id', 'amount'], keep='first')
    
    # Add column names to df_original_clean if not already present
    if 'name' not in df_original_clean.columns and 'name' in high_very_high_clean.columns:
        names_df = high_very_high_clean[['transaction_id', 'id', 'name']].drop_duplicates(subset=['transaction_id'], keep='first')
        df_original_clean = df_original_clean.merge(names_df[['transaction_id', 'name']], on='transaction_id', how='left')
    
    # Merge with original data to get all features for explanation
    merge_cols = ['transaction_id', 'id', 'amount',
                  'hour', 'is_extreme_late_night', 'is_business_hours', 'is_weekend',
                  'is_very_high_value', 'exceeds_daily_limit', 'is_near_max_limit',
                  'is_very_rapid_trx', 'is_high_velocity_',
                  'has_same_amount_pattern', 'consecutive_same_amount', 'is_limit_gaming']
    
    # Add names to merge_cols if available
    if 'name' in df_original_clean.columns:
        merge_cols.extend(['name'])
    
    high_risk_report = high_very_high_clean.merge(
        df_original_clean[merge_cols],
        on=['transaction_id', 'id', 'amount'],
        how='left',
        suffixes=('', '_orig')
    )
    
    # Remove any duplicates that might have occurred during merge
    high_risk_report = high_risk_report.drop_duplicates(subset=['transaction_id'], keep='first')
    
    # Debug: Show available columns for explanation
    print(f"\n📋 Columns available for explanation generation:")
    feature_cols = [col for col in high_risk_report.columns if col.startswith('is_') or col.endswith('_rate') or 
                    col.endswith('_count') or col.endswith('_min') or col == 'hour' or col == 'amount']
    print(f"   Found {len(feature_cols)} feature columns")
    if len(feature_cols) > 0:
        print(f"   Sample features: {', '.join(feature_cols[:10])}")
    else:
        print("   ⚠️ Warning: No feature columns found! Merge may have failed.")
        print(f"   Available columns: {', '.join(high_risk_report.columns[:15])}")
    
    # Generate explanation function with USER-FRIENDLY comprehensive checks
    def generate_explanation(row):
        reasons = []
        
        # Helper untuk nama hari
        def get_day_name(is_weekend):
            if is_weekend:
                return "Sabtu/Minggu"
            return "hari kerja (Senin-Jumat)"
        
        # Score-based (always included)
        if row['anomaly_score'] >= threshold_high_risk * 1.2:
            reasons.append(f"Score anomali sangat tinggi ({row['anomaly_score']:.3f})")
        elif row['anomaly_score'] >= threshold_high_risk:
            reasons.append(f"Score anomali tinggi ({row['anomaly_score']:.3f})")
        
        # Time-based patterns
        hour = row.get('hour', -1)
        is_weekend = row.get('is_weekend', 0) == 1
        is_business_hours = row.get('is_business_hours', 0) == 1
        is_peak_hours = row.get('is_peak_hours', 0) == 1
        day_name = get_day_name(is_weekend)
        
        # PERBAIKAN: Hanya flag transaksi dini hari yang benar-benar mencurigakan
        if row.get('is_extreme_late_night', 0) == 1 or (0 <= hour <= 4):
            reasons.append(f"Transaksi dini hari ekstrem (jam {int(hour):02d}:00 pada {day_name}) - waktu yang sangat tidak umum untuk aktivitas normal")
        elif (hour >= 22 or hour <= 5) and not is_peak_hours:
            reasons.append(f"Transaksi di jam tidak umum (jam {int(hour):02d}:00 pada {day_name}) - di luar jam operasional bisnis normal")
        
        # PERBAIKAN 4: Weekend dengan detail hari dan jam
        if is_weekend:
            if not is_business_hours and not is_peak_hours:
                reasons.append(f"Transaksi di akhir pekan ({day_name}) di luar jam bisnis (08:00-20:00) dan jam ramai (11:00-14:00, 18:00-21:00) pada jam {int(hour):02d}:00 - pola tidak wajar untuk transaksi retail")
            elif (0 <= hour <= 5) or (hour >= 22):
                reasons.append(f"Transaksi di akhir pekan ({day_name}) pada jam tidak wajar (jam {int(hour):02d}:00) - aktivitas mencurigakan")
        
        # Amount-based patterns
        amount = row.get('amount', 0)
        if row.get('exceeds_daily_limit', 0) == 1 or amount > 10000000:
            reasons.append(f"Melebihi limit harian (nilai: Rp {amount:,.0f}) - potensi pencucian uang atau transaksi ilegal")
        elif row.get('is_very_high_value', 0) == 1 or amount > 50000000:
            reasons.append(f"Nilai sangat tinggi tidak wajar (Rp {amount:,.0f}) - jauh di atas rata-rata transaksi retail")
        elif amount > 5000000:
            reasons.append(f"Nilai transaksi besar (Rp {amount:,.0f}) - perlu verifikasi kewajaran transaksi")
        
        if row.get('is_near_max_limit', 0) == 1:
            reasons.append("Mendekati batas maksimal limit harian - potensi upaya memaksimalkan transaksi sebelum terdeteksi")
        
        if row.get('is_limit_gaming', 0) == 1 or amount in [9999999, 9999990, 9999900]:
            reasons.append(f"Pola limit gaming terdeteksi (Rp {amount:,}) - upaya sengaja mendekati limit tanpa melampaui untuk menghindari deteksi")
        
        # Velocity-based patterns
        if row.get('is_very_rapid_trx', 0) == 1:
            reasons.append("Transaksi sangat cepat (<10 detik antar transaksi) - tidak mungkin dilakukan secara normal, indikasi otomasi atau cloning kartu")
        
        if row.get('is_high_velocity', 0) == 1:
            reasons.append("Frekuensi sangat tinggi (5+ transaksi/menit) - kecepatan tidak wajar untuk proses pembayaran normal, kemungkinan fraud terorganisir")
        
        # Check for rapid transaction indicators from other columns
        if row.get('trx_1min', 0) >= 3:
            reasons.append(f"Banyak transaksi dalam 1 menit ({int(row.get('trx_1min', 0))} transaksi) - kecepatan melampaui kemampuan kasir normal, indikasi sistem otomatis atau multiple device fraud")
        
        if row.get('trx_5min', 0) >= 10:
            reasons.append(f"Lonjakan transaksi dalam 5 menit ({int(row.get('trx_5min', 0))} transaksi) - volume tidak proporsional dengan ukuran normal")
        
        # Pattern-based detection
        if row.get('has_same_amount_pattern', 0) == 1:
            reasons.append("Pola nominal sama berulang dalam periode singkat - indikasi testing kartu curian atau transaksi fiktif berulang")
        
        if row.get('consecutive_same_amount', 0) == 1:
            reasons.append("3+ transaksi berurutan dengan nominal identik - sangat tidak wajar dalam retail normal, kemungkinan besar fraud terstruktur")
        
        # Type patterns
        type = row.get('type', '')
        if type == 'DYNAMIC' and row.get('suspicious_dynamic', 0) > 0:
            reasons.append("Pola Dynamic mencurigakan - penggunaan dinamis dengan pola tidak sesuai prosedur bisnis normal")
        
        # Behavior patterns
        if row.get('success_rate', 1) < 0.7:
            success_rate = row.get('success_rate', 0) * 100
            reasons.append(f"Success rate rendah ({success_rate:.1f}%) - banyak transaksi gagal menunjukkan aktivitas mencoba-coba atau testing fraud")
        
        # PERBAIKAN 1: Deviation from normal patterns - dijelaskan lebih detail
        if row.get('amount_deviation_from_mean', 0) > 3:
            reasons.append(f"Nilai transaksi menyimpang ekstrem dari rata-rata ini (>3x standar deviasi) - biasanya melakukan transaksi dengan nilai jauh lebih rendah/tinggi, pola ini sangat tidak sesuai karakteristik normal")
        
        # PERBAIKAN 2: Rolling window anomalies - dijelaskan lebih detail
        if row.get('is_spike_vs_rolling_50', 0) == 1:
            reasons.append("Lonjakan tidak wajar dibanding pola historis 50 transaksi terakhir - volume atau nilai transaksi tiba-tiba melonjak tanpa alasan bisnis yang jelas, tidak konsisten dengan tren normal")
        
        # Payment method unusual
        payment = row.get('payment_method', '')
        if payment and row.get('is_unusual_payment', 0) == 1:
            reasons.append(f"Metode pembayaran tidak biasa untuk ini ({payment}) - biasanya tidak menggunakan metode ini, potensi testing fraud dengan berbagai channel")
        
        # PERBAIKAN 3: Fallback - dijelaskan lebih user-friendly
        if len(reasons) <= 1:
            # Add general anomaly indicators based on risk level
            if row.get('risk_level') == 'very_high_risk':
                reasons.append("Kombinasi multiple faktor risiko sangat tinggi terdeteksi - transaksi ini memiliki banyak indikator fraud yang saling menguatkan")
                reasons.append("Pola transaksi menyimpang signifikan dari baseline normal - karakteristik transaksi (waktu, nilai, frekuensi, lokasi) sangat berbeda dari pola historis yang sehat, menunjukkan perubahan perilaku drastis yang mencurigakan")
            else:
                reasons.append("Pola transaksi tidak sesuai dengan karakteristik normal - berdasarkan analisis fitur statistik (waktu transaksi, nilai rata-rata, frekuensi, metode pembayaran, success rate), transaksi ini memiliki profil yang berbeda dari 95% transaksi normal")
                reasons.append("Terdeteksi anomali pada fitur statistik transaksi - model AI mendeteksi penyimpangan pada kombinasi fitur seperti: pola waktu transaksi, distribusi nilai, kecepatan antar transaksi, rasio sukses/gagal, yang secara statistik berada di luar range normal")
        
        return " | ".join(reasons)
    
    print("\nGenerating explanations...")
    high_risk_report['explanation'] = high_risk_report.apply(generate_explanation, axis=1)
    
    # Debug: Check explanation quality
    print(f"✓ Explanation column created")
    
    # Count explanations by number of reasons
    explanation_lengths = high_risk_report['explanation'].str.split(' | ').apply(len)
    print(f"\n📊 Explanation Quality Statistics:")
    print(f"   Average reasons per transaction: {explanation_lengths.mean():.1f}")
    print(f"   Min reasons: {explanation_lengths.min()}")
    print(f"   Max reasons: {explanation_lengths.max()}")
    
    # Show explanations with only 1 reason (problematic)
    single_reason = high_risk_report[explanation_lengths == 1]
    if len(single_reason) > 0:
        print(f"\n⚠️ Warning: {len(single_reason)} transactions have only 1 reason (score only)")
        print(f"   Sample: {single_reason['explanation'].iloc[0]}")
    
    # Show sample of good explanations
    print(f"\n📝 Sample explanations (first 3):")
    for i, exp in enumerate(high_risk_report['explanation'].head(3), 1):
        print(f"   {i}. {exp[:200]}{'...' if len(exp) > 200 else ''}")
    
    # Create final report with proper column order
    # Start with base columns
    base_cols = ['transaction_id', 'id']
    
    # Add name if available
    if 'name' in high_risk_report.columns:
        base_cols.append('name')
    
    # Add id columns
    base_cols.append('id')
    
    # Add name if available
    if 'name' in high_risk_report.columns:
        base_cols.append('name')
    
    # Add remaining columns
    base_cols.extend(['transaction_timestamp', 'transaction_date', 'amount', 
                      'anomaly_score', 'risk_level', 'status_name', 'explanation'])
    
    # Select only existing columns
    final_report_cols = [col for col in base_cols if col in high_risk_report.columns]
    final_report = high_risk_report[final_report_cols].copy()
    
    # Remove duplicates
    final_report = final_report.drop_duplicates(subset=['transaction_id'], keep='first')
    
    # Sort by risk level and anomaly score
    risk_order = {'very_high_risk': 0, 'high_risk': 1}
    final_report['risk_order'] = final_report['risk_level'].map(risk_order)
    final_report = final_report.sort_values(['risk_order', 'anomaly_score'], ascending=[True, False])
    final_report.drop('risk_order', axis=1, inplace=True)
    
    # Final check for duplicates before ranking
    duplicates_count = final_report.duplicated(subset=['transaction_id']).sum()
    if duplicates_count > 0:
        print(f"⚠️ Removing {duplicates_count} duplicate transactions...")
        final_report = final_report.drop_duplicates(subset=['transaction_id'], keep='first')
    
    final_report['priority_rank'] = range(1, len(final_report) + 1)
    
    print(f"\n✅ Report generated: {len(final_report):,} unique transactions (no duplicates)")
    print(f"   - VERY HIGH RISK: {(final_report['risk_level'] == 'very_high_risk').sum():,}")
    print(f"   - HIGH RISK: {(final_report['risk_level'] == 'high_risk').sum():,}")
    
    # Display top 20
    print(f"\n📋 TOP 20 PRIORITY TRANSACTIONS:")
    display_cols = ['priority_rank', 'transaction_id', 'id']
    if 'name' in final_report.columns:
        display_cols.append('name')
    display_cols.extend(['id'])
    if 'name' in final_report.columns:
        display_cols.append('name')
    display_cols.extend(['amount', 'anomaly_score', 'risk_level'])
    
    print(final_report[display_cols].head(20).to_string(index=False))
    
    print(f"\n\n📝 SAMPLE EXPLANATIONS (Top 5):")
    for idx, row in final_report.head(5).iterrows():
        name = row.get('name', f"{row['id']}")
        
        print(f"\n{row['priority_rank']}. Transaction: {row['transaction_id']}")
        print(f"   Name: {name}")
        print(f"   Amount: Rp {row['amount']:,.0f}")
        print(f"   Anomaly Score: {row['anomaly_score']:.6f}")
        print(f"   Risk Level: {row['risk_level'].upper()}")
        print(f"   Explanation: {row['explanation']}")
    
    # Reorder columns for export (name next to id)
    export_cols = ['priority_rank', 'transaction_id', 'id']
    if 'name' in final_report.columns:
        export_cols.append('name')
    export_cols.append('id')
    if 'name' in final_report.columns:
        export_cols.append('name')
    
    # Add remaining columns
    remaining_cols = [col for col in final_report.columns if col not in export_cols]
    export_cols.extend(remaining_cols)
    
    final_report_export = final_report[export_cols].copy()
    
    # Save complete report with proper column order
    final_report_export.to_csv('realtime_complete_high_risk_report.csv', index=False)
    print(f"\n✅ Saved: realtime_complete_high_risk_report.csv")
    print(f"   Columns order: {', '.join(export_cols[:10])}...")
    
    # Export to Excel with multiple sheets
    with pd.ExcelWriter('realtime_high_risk_analysis.xlsx', engine='openpyxl') as writer:
        # Sheet 1: Complete transactions with proper column order
        final_report_export.to_excel(writer, sheet_name='All High Risk Transactions', index=False)
        
        # Sheet 2: summary
        if len(agg) > 0:
            agg.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 3: Top 10 detail
        if len(agg) >= 10:
            top_10 = agg.head(10)
            for idx, row in top_10.iterrows():
                id = row['id']
                
                trx = final_report_export[(final_report_export['id'] == id)].copy()
                
                # Ensure no duplicates in detail sheet
                trx = trx.drop_duplicates(subset=['id'], keep='first')
                
                sheet_name = f"M{int(id)}_O{int(id)}"[:31]
                trx.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"✅ Saved: realtime_high_risk_analysis.xlsx (multi-sheet)")
    
else:
    print("\n✅ EXCELLENT! No HIGH RISK or VERY HIGH RISK transactions found!")
    print("All transactions are operating normally.")

print("=" * 100)


## 5.2 Final Summary

In [ ]:
import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
import os
from collections import Counter

print("\n" + "=" * 100)
print("📧 SENDING EMAIL NOTIFICATION")
print("=" * 100)

# Check if files exist
required_files = [
    'realtime_behavioral_profile.csv',
    'realtime_high_risk_analysis.xlsx'
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    print(f"\n⚠️ Missing files: {', '.join(missing_files)}")
    print("Please run all previous cells to generate the reports first.")
else:
    # Email configuration
    email_sender = 'mail@wew.co.id'
    email_password = 'app&*(mail'
    email_receivers = ['wew@wew.co.id']
    
    # Get summary statistics
    total_trx = len(results_df)
    risk_counts = results_df['risk_level'].value_counts()
    
    # PERBAIKAN 1: Gabungkan suspicious ke normal
    normal_count = risk_counts.get('normal', 0) + risk_counts.get('suspicious', 0)
    high_risk_count = risk_counts.get('high_risk', 0)
    very_high_risk_count = risk_counts.get('very_high_risk', 0)
    anomaly_count = high_risk_count + very_high_risk_count
    
    # Get transaction time range
    min_time = results_df['transaction_timestamp'].min()
    max_time = results_df['transaction_timestamp'].max()
    
    # PERBAIKAN 3: Analisis pola anomali yang paling sering
    anomaly_patterns = {}
    top_patterns_html = ""
    
    if len(high_very_high) > 0 and 'explanation' in final_report.columns:
        # Extract all reasons from explanations
        all_reasons = []
        for exp in final_report['explanation']:
            reasons = exp.split(' | ')
            # Skip score-based reason (first one) and filter empty
            all_reasons.extend([r.strip() for r in reasons[1:] if r.strip()])
        
        # Count frequency
        pattern_counter = Counter(all_reasons)
        top_patterns = pattern_counter.most_common(5)
        
        if top_patterns:
            top_patterns_html = """
        <div class="info">
            <h2>🔍 Pola Anomali Yang Terdeteksi</h2>
            <p>Berdasarkan analisis <strong>{:,}</strong> transaksi berisiko tinggi, pola anomali yang paling sering ditemukan adalah:</p>
            <div class="pattern-list">
            """.format(anomaly_count)
            
            for idx, (pattern, count) in enumerate(top_patterns, 1):
                pct = (count / anomaly_count * 100)
                top_patterns_html += f"""
                <div class="pattern-item">
                    <strong>{idx}. {pattern}</strong><br>
                    <span style="color: #7f8c8d;">Ditemukan pada {count:,} transaksi ({pct:.1f}%)</span>
                </div>
                """
            
            top_patterns_html += """
            </div>
            <p style="margin-top: 15px;"><em>💡 Rekomendasi: Fokuskan investigasi pada dengan pola anomali yang paling sering muncul.</em></p>
        </div>
            """
    
    # Create message
    msg = MIMEMultipart('alternative')
    msg['From'] = email_sender
    msg['To'] = ', '.join(email_receivers)
    msg['Subject'] = f'📊 FDS QRIS Report - {datetime.now().strftime("%Y-%m-%d %H:%M")} - {high_risk_count} High Risk, {very_high_risk_count} Very High Risk'
    
    # PERBAIKAN 2: Warna lebih profesional (tidak menakutkan)
    html_body = f"""
    <html>
    <head>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                color: #2c3e50;
                max-width: 1200px;
                margin: 0 auto;
                background-color: #f8f9fa;
            }}
            .header {{
                background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
                color: white;
                padding: 25px;
                text-align: center;
                border-radius: 8px 8px 0 0;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }}
            .header h1 {{
                margin: 0;
                font-size: 24px;
            }}
            .header p {{
                margin: 5px 0 0 0;
                font-size: 14px;
                opacity: 0.9;
            }}
            .summary {{
                background-color: white;
                border-left: 4px solid #3498db;
                padding: 20px;
                margin: 20px;
                border-radius: 4px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.05);
            }}
            .info {{
                background-color: #e8f4f8;
                border-left: 4px solid #5dade2;
                padding: 20px;
                margin: 20px;
                border-radius: 4px;
            }}
            .critical {{
                background-color: white;
                border-left: 4px solid #e67e22;
                padding: 20px;
                margin: 20px;
                border-radius: 4px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.05);
            }}
            table {{
                border-collapse: collapse;
                width: 100%;
                margin: 15px 0;
                background-color: white;
                box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            }}
            th {{
                background-color: #34495e;
                color: white;
                padding: 12px;
                text-align: left;
                font-weight: 600;
                font-size: 13px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ecf0f1;
                font-size: 13px;
            }}
            tr:hover {{
                background-color: #f8f9fa;
            }}
            .risk-very-high {{
                color: #c0392b;
                font-weight: bold;
                padding: 4px 8px;
                background-color: #fadbd8;
                border-radius: 3px;
            }}
            .risk-high {{
                color: #e67e22;
                font-weight: bold;
                padding: 4px 8px;
                background-color: #fdeaa8;
                border-radius: 3px;
            }}
            .footer {{
                background-color: #ecf0f1;
                padding: 20px;
                text-align: center;
                font-size: 12px;
                color: #7f8c8d;
                border-radius: 0 0 8px 8px;
                margin-top: 20px;
            }}
            .pattern-list {{
                background-color: white;
                padding: 15px;
                border-radius: 4px;
                margin: 15px 0;
            }}
            .pattern-item {{
                padding: 12px;
                margin: 8px 0;
                border-left: 3px solid #5dade2;
                background-color: #f8f9fa;
                border-radius: 3px;
            }}
            .status-badge {{
                display: inline-block;
                padding: 6px 12px;
                border-radius: 4px;
                font-weight: bold;
                font-size: 12px;
            }}
            .badge-safe {{
                background-color: #d5f4e6;
                color: #27ae60;
            }}
            .badge-attention {{
                background-color: #fdeaa8;
                color: #e67e22;
            }}
            .badge-urgent {{
                background-color: #fadbd8;
                color: #c0392b;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>📊 FDS QRIS Real-Time Monitoring Report</h1>
            <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
        
        <div class="summary">
            <h2 style="color: #2c3e50; margin-top: 0;">📈 Summary Hasil Prediksi</h2>
            <p><strong>Periode Transaksi:</strong> {min_time.strftime('%Y-%m-%d %H:%M:%S')} s/d {max_time.strftime('%Y-%m-%d %H:%M:%S')}</p>
            <p><strong>Total Transaksi Dianalisis:</strong> {total_trx:,}</p>
            
            <table style="margin-top: 20px;">
                <tr style="background-color: #d5f4e6;">
                    <td><span class="status-badge badge-safe">✅ NORMAL TRANSACTIONS</span></td>
                    <td style="text-align: right;"><strong>{normal_count:,}</strong></td>
                    <td style="text-align: right;">{(normal_count/total_trx*100):.1f}%</td>
                </tr>
                <tr style="background-color: #fdeaa8;">
                    <td><span class="status-badge badge-attention">⚠️ HIGH RISK</span></td>
                    <td style="text-align: right;"><strong>{high_risk_count:,}</strong></td>
                    <td style="text-align: right;">{(high_risk_count/total_trx*100):.1f}%</td>
                </tr>
                <tr style="background-color: #fadbd8;">
                    <td><span class="status-badge badge-urgent">🚨 VERY HIGH RISK</span></td>
                    <td style="text-align: right;"><strong>{very_high_risk_count:,}</strong></td>
                    <td style="text-align: right;">{(very_high_risk_count/total_trx*100):.1f}%</td>
                </tr>
                <tr style="background-color: #e8f4f8; font-weight: bold;">
                    <td><strong>📊 TOTAL PERLU INVESTIGASI</strong></td>
                    <td style="text-align: right;"><strong>{anomaly_count:,}</strong></td>
                    <td style="text-align: right;">{(anomaly_count/total_trx*100):.1f}%</td>
                </tr>
            </table>
        </div>
        
        {top_patterns_html}
    """
    
    # Add HIGH RISK table if available
    if len(high_very_high) > 0 and len(agg) > 0:
        high_very_high = agg[
            agg['risk_category'].isin(['HIGH_RISK', 'VERY_HIGH_RISK'])
        ].head(20)
        
        html_body += f"""
        <div class="critical">
            <h2 style="color: #2c3e50; margin-top: 0;">🎯 Top 20 Yang Perlu Investigasi</h2>
            <p>Daftar dengan transaksi berisiko tinggi yang memerlukan perhatian:</p>
            <table>
                <tr>
                    <th>Rank</th>
                    <th>ID</th>
                    <th>Name</th>
                    <th>Total Trx</th>
                    <th>Avg Anomaly Score</th>
                    <th>Very High Risk</th>
                    <th>Risk Category</th>
                </tr>
        """
        
        for idx, row in enumerate(high_very_high.itertuples(), 1):
            risk_class = 'risk-very-high' if row.risk_category == 'VERY_HIGH_RISK' else 'risk-high'
            risk_display = 'VERY HIGH' if row.risk_category == 'VERY_HIGH_RISK' else 'HIGH'
            
            html_body += f"""
                <tr>
                    <td>{idx}</td>
                    <td>{row.id}</td>
                    <td>{row.name}</td>
                    <td>{row.total_trx:,}</td>
                    <td>{row.avg_anomaly_score:.2f}</td>
                    <td>{row.very_high_risk_count:,}</td>
                    <td><span class="{risk_class}">{risk_display}</span></td>
                </tr>
            """
        
        html_body += """
            </table>
        </div>
        """
    elif len(high_very_high) == 0:
        html_body += """
        <div class="summary" style="background-color: #d5f4e6; border-left: 4px solid #27ae60;">
            <h2 style="color: #27ae60; margin-top: 0;">✅ Status: AMAN</h2>
            <p>Tidak ada transaksi dengan kategori HIGH RISK atau VERY HIGH RISK yang terdeteksi pada periode ini.</p>
            <p><em>Sistem akan terus memantau transaksi secara real-time.</em></p>
        </div>
        """
    
    html_body += f"""
        <div class="footer">
            <p><strong>📎 File Terlampir:</strong></p>
            <ul style="list-style: none; padding: 0;">
                <li>📄 realtime_behavioral_profile.csv - Profil behavioral</li>
                <li>📄 realtime_high_risk_analysis.xlsx - Analisis lengkap transaksi berisiko</li>
            </ul>
            <hr style="border: none; border-top: 1px solid #bdc3c7; margin: 20px 0;">
            <p style="margin: 5px 0;">FDS Monitoring System</p>
            <p style="margin: 5px 0; font-size: 11px;">Email ini digenerate secara otomatis. Untuk informasi lebih lanjut, silakan hubungi tim FDS.</p>
        </div>
    </body>
    </html>
    """
    
    # Attach HTML body
    msg.attach(MIMEText(html_body, 'html'))
    
    # Attach files
    for filename in required_files:
        if os.path.exists(filename):
            with open(filename, 'rb') as attachment:
                part = MIMEBase('application', 'octet-stream')
                part.set_payload(attachment.read())
                encoders.encode_base64(part)
                part.add_header(
                    'Content-Disposition',
                    f'attachment; filename= {filename}'
                )
                msg.attach(part)
    
    # Send email
    try:
        # Connect to SMTP server
        smtp_server = 'mail.wew.co.id'
        smtp_port = 25
        
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.ehlo()
        
        # Login and send
        server.login(email_sender, email_password)
        server.sendmail(email_sender, email_receivers, msg.as_string())
        server.quit()
        
        print(f"\n✅ Email berhasil dikirim ke: {', '.join(email_receivers)}")
        print(f"📊 Summary:")
        print(f"   - Total Transaksi: {total_trx:,}")
        print(f"   - Normal (incl. Suspicious): {normal_count:,} ({(normal_count/total_trx*100):.1f}%)")
        print(f"   - High Risk: {high_risk_count:,} ({(high_risk_count/total_trx*100):.1f}%)")
        print(f"   - Very High Risk: {very_high_risk_count:,} ({(very_high_risk_count/total_trx*100):.1f}%)")
        print(f"   - Total Perlu Investigasi: {anomaly_count:,} ({(anomaly_count/total_trx*100):.1f}%)")
        
        if anomaly_patterns:
            print(f"\n🔍 Top 3 Pola Anomali:")
            for idx, (pattern, count) in enumerate(list(pattern_counter.most_common(3)), 1):
                print(f"   {idx}. {pattern}: {count} transaksi")
        
    except Exception as e:
        print(f"\n❌ Error saat mengirim email: {str(e)}")
        import traceback
        traceback.print_exc()


In [ ]:
print("\n" + "=" * 100)
print("🎯 FINAL SUMMARY - REAL-TIME FRAUD DETECTION")
print("=" * 100)

print(f"\n📊 DATA PROCESSED:")
print(f"   Total Transactions: {len(results_df):,}")
print(f"   Date Range: {START_DATE} to {END_DATE}")
print(f"   Unique: {results_df['id'].nunique():,}")

print(f"\n⚠️ RISK DISTRIBUTION:")
risk_summary = results_df['risk_level'].value_counts()
for level in ['very_high_risk', 'high_risk', 'suspicious', 'normal']:
    count = risk_summary.get(level, 0)
    pct = count / len(results_df) * 100
    print(f"   {level.upper()}: {count:,} ({pct:.2f}%)")

if len(high_very_high) > 0:
    print(f"\n🚨 HIGH PRIORITY ALERTS:")
    print(f"   HIGH + VERY HIGH RISK Transactions: {len(high_very_high):,}")
    print(f"   Affected: {high_very_high['id'].nunique()}")
    print(f"   Affected: {high_very_high.groupby(['id']).ngroups}")
    print(f"   Total Amount at Risk: Rp {high_very_high['amount'].sum():,.0f}")
    
    print(f"\n📁 OUTPUT FILES GENERATED:")
    files = [
        'realtime_high_risk.csv',
        'realtime_high_risk_transactions.csv',
        'realtime_spike_detection.csv',
        'realtime_behavioral_profile.csv',
        'realtime_complete_high_risk_report.csv',
        'realtime_high_risk_analysis.xlsx',
        'realtime_top_suspicious.png'
    ]
    for i, f in enumerate(files, 1):
        print(f"   {i}. {f}")
else:
    print(f"\n✅ EXCELLENT NEWS!")
    print(f"   No HIGH RISK or VERY HIGH RISK transactions detected!")
    print(f"   All transactions are operating normally.")

print(f"\n🔧 MODEL USED:")
print(f"   Algorithm: Isolation Forest")
print(f"   Contamination: {model.contamination}")
print(f"   Features: {len(feature_names)}")
print(f"   Thresholds: Suspicious={threshold_suspicious:.4f}, High Risk={threshold_high_risk:.4f}")

print(f"\n🎯 KEY INSIGHTS:")
if len(high_very_high) > 0:
    print(f"   1. Anomaly Score: {len(agg)} dengan HIGH/VERY HIGH RISK")
    print(f"   3. Transaction Anomaly Score: {len(high_very_high):,} transaksi HIGH/VERY HIGH RISK")
    print(f"   4. Spike Detection: {len(spike_days)} hari dengan spike pattern" if 'spike_days' in locals() else "")
    print(f"   5. Behavioral Profiles: Tersedia untuk semua high risk entities")
    print(f"   6. Complete Report: Dengan penjelasan lengkap per transaksi")
    print(f"   7. Nama Lengkap: Semua output include name")
    
    print(f"\n⚡ NEXT ACTIONS:")
    print(f"   1. Review realtime_high_risk_analysis.xlsx untuk detail lengkap")
    print(f"   2. Prioritas: Check Top 10 di realtime_high_risk.csv")
    print(f"   3. Investigate transaksi VERY HIGH RISK terlebih dahulu")
    print(f"   4. Monitor spike patterns di realtime_spike_detection.csv")
else:
    print(f"   ✅ No anomalies detected - system is healthy!")

print("\n" + "=" * 100)
print("✅ REAL-TIME FRAUD DETECTION ANALYSIS COMPLETED!")
print("=" * 100)